In [26]:
import pandas as pd
import subprocess
subprocess.run(['pip', 'install', 'xlrd'], capture_output=True)

# Load the XLS file
df = pd.read_excel(r'D:\Data\Asset Allocation Project\list_of_etfs_and_etps_securities_6 (1).xls', 
                   engine='xlrd', header=2)
# Set correct headers
df.columns = df.iloc[0]
df = df.drop(index=0).reset_index(drop=True)

# Keep only columns we need
etfs = df[['Mnemonic', 'Long Name', 'Issuer Name', 'Security Type']].copy()

# Add .L suffix for yfinance
etfs['Ticker'] = etfs['Mnemonic'].astype(str) + '.L'

# Drop any rows with missing names
etfs = etfs.dropna(subset=['Long Name', 'Mnemonic'])

# Save as CSV
etfs.to_csv(r'D:\Data\Asset Allocation Project\lse_etfs.csv', index=False)

print(etfs['Issuer Name'].value_counts())


# Assign Asset Class based on Long Name keywords
def map_asset_class(name):
    name_upper = str(name).upper()
    
    # Leveraged/short products first - override everything
    if any(x in name_upper for x in ['LEVERAGE', 'SHORT', '2X', '3X', 'DAILY']): 
        return 'Alternatives'
    
    # Fixed income
    if any(x in name_upper for x in ['GILT', 'BOND', 'TREASURY', 'SOVEREIGN', 'CREDIT', 'INFLATION', 'FIXED INCOME']):
        return 'Fixed Income'
    
    # Commodities and alternatives
    if any(x in name_upper for x in ['GOLD', 'SILVER', 'COMMODITY', 'OIL', 'METAL', 'MINING', 'ENERGY', 'REIT', 'PROPERTY', 'REAL ESTATE']):
        return 'Alternatives'
    
    # Cash
    if any(x in name_upper for x in ['OVERNIGHT', 'MONEY MARKET', 'CASH']):
        return 'Cash'
    
    # UK Equity
    if any(x in name_upper for x in ['FTSE 100', 'FTSE100', 'UK EQUITY', 'UK STOCK']):
        return 'UK Equity'
    
    # Global Equity
    if any(x in name_upper for x in ['EQUITY', 'MSCI', 'S&P', 'FTSE', 'NASDAQ', 'STOCK', 'SHARES', 'WORLD', 'GLOBAL', 'EMERGING']):
        return 'Global Equity'
    
    # Nothing matched
    return 'N/A'

etfs['Asset_Class'] = etfs['Long Name'].apply(map_asset_class)

print(etfs['Asset_Class'].value_counts())
print(f"\nN/A count: {(etfs['Asset_Class'] == 'N/A').sum()}")

# Filter to only N/A rows to review what's not been categorised
na_etfs = etfs[etfs['Asset_Class'] == 'N/A']

print(f"Total N/A: {len(na_etfs)}")
print(na_etfs[['Mnemonic', 'Long Name', 'Asset_Class']].to_string())
# Expanded keyword categorisation with more terms from the N/A list
def map_asset_class(name):
    name_upper = str(name).upper()
    
    # Leveraged/short first
    if any(x in name_upper for x in ['LEVERAGE', '2X', '3X', 'DAILY SHORT', 'DAILY LONG']): 
        return 'Alternatives'
    
    # Commodities - expanded
    if any(x in name_upper for x in [
        'GOLD', 'SILVER', 'COPPER', 'NICKEL', 'ZINC', 'ALUMINIUM', 'PLATINUM', 
        'PALLADIUM', 'RHODIUM', 'TIN', 'LEAD', 'COMMODITY', 'COMMODITIES', 
        'AGRICULTURE', 'AGRICULTURAL', 'GRAINS', 'WHEAT', 'CORN', 'SUGAR', 
        'COFFEE', 'COCOA', 'COTTON', 'SOYBEANS', 'SOFTS', 'NATURAL GAS', 
        'GASOLINE', 'PETROLEUM', 'OIL', 'BRENT', 'WTI', 'ENERGY',
        'METAL', 'MINING', 'LIVESTOCK', 'CATTLE', 'HOGS',
        'REIT', 'PROPERTY', 'REAL ESTATE', 'INFRASTRUCTURE', 'TIMBER',
        'BLOCKCHAIN', 'CRYPTO', 'CANNABIS', 'MLP', 'ROBOTICS',
        'MANAGED FUTURES', 'ROLL SELECT', 'CMCI', 'CRB', 'RICI',
        'CYBER', 'BATTERY', 'ECOMMERCE', 'PHARMA', 'HEALTHCARE INNOV',
        'ARTIFICIAL INTELLIGENCE', 'CLOUD', 'INTERNET', 'TECHNOLOGY INNOV'
    ]):
        return 'Alternatives'
    
    # Fixed income - expanded  
    if any(x in name_upper for x in [
        'GILT', 'BOND', 'TREASURY', 'TREASURIES', 'SOVEREIGN', 'CREDIT',
        'INFLATION', 'TIPS', 'CORPORATE', 'CORP BD', 'HIGH YIELD', 'HY',
        'FLOATING RATE', 'GOVERNMENT', 'GOV BD', 'AGGREGATE', 'AGG',
        'IBOXX', 'ITRAXX', 'MORTGAGE', 'MTGE', 'DURATION', 'FIXED INCOME',
        'ULTRASHORT', 'ULTRASRT', 'TBILL', 'T-BILL'
    ]):
        return 'Fixed Income'
    
    # Cash
    if any(x in name_upper for x in ['OVERNIGHT', 'MONEY MARKET', 'CASH', 'LIQUIDITY']):
        return 'Cash'
    
    # UK Equity
    if any(x in name_upper for x in ['FTSE 100', 'FTSE100', 'UK EQ', 'UNITED KINGDOM']):
        return 'UK Equity'
    
    # Global Equity - expanded
    if any(x in name_upper for x in [
        'EQUITY', 'EQUITIES', 'MSCI', 'S&P', 'FTSE', 'NASDAQ', 'STOXX',
        'STOCK', 'SHARES', 'WORLD', 'GLOBAL', 'EMERGING', 'RUSSELL',
        'NIKKEI', 'TOPIX', 'DAX', 'CAC', 'JAPAN', 'CHINA', 'INDIA',
        'EUROPE', 'ASIA', 'DIVIDEND', 'INCOME', 'GROWTH', 'VALUE',
        'MOMENTUM', 'QUALITY', 'FACTOR', 'MINIMUM VARIANCE', 'MIN VAR',
        'SMALL CAP', 'MID CAP', 'LARGE CAP', 'DOW JONES', 'NIFTY',
        'BIOTECH', 'BANKS', 'HEALTHCARE', 'UTILITIES', 'INDUSTRIALS',
        'CONSUMER', 'COMMUNICATIONS', 'MATERIALS', 'MORNINGSTAR'
    ]):
        return 'Global Equity'
    
    return 'N/A'

etfs['Asset_Class'] = etfs['Long Name'].apply(map_asset_class)

print(etfs['Asset_Class'].value_counts())
print(f"\nRemaining N/A: {(etfs['Asset_Class'] == 'N/A').sum()}")

def map_asset_class_v3(name):
    name_upper = str(name).upper()

    # Leveraged/short first
    if any(x in name_upper for x in ['LEVERAGE', '2X', '3X', 'DAILY SHORT', 'DAILY LONG']):
        return 'Alternatives'

    # Commodities - include abbreviations
    if any(x in name_upper for x in [
        'GOLD', 'SILVER', 'COPPER', 'NICKEL', 'ZINC', 'ALUMINIUM', 'PLATINUM',
        'PALLADIUM', 'RHODIUM', 'TIN', 'LEAD', 'COMMODITY', 'COMMODITIES', 'COMMO',
        'AGRICULTURE', 'GRAINS', 'WHEAT', 'CORN', 'SUGAR', 'COFFEE', 'COCOA',
        'COTTON', 'SOYBEANS', 'SOFTS', 'NATURAL GAS', 'GASOLINE', 'PETROLEUM',
        'OIL', 'BRENT', 'WTI', 'METAL', 'MINING', 'LIVESTOCK', 'CATTLE', 'HOGS',
        'REIT', 'PROPERTY', 'PPTY', 'PROP ', 'REAL EST', 'INFRASTRUCTURE', 'TIMBER',
        'BLOCKCHAIN', 'CANNABIS', 'MLP', 'ROBOTICS', 'MANAGED FUTURES',
        'CMCI', 'CRB', 'RICI', 'BCOM', 'GSCI',
        'CYBER', 'BATTERY', 'ECOMMERCE', 'PHARMA',
        'ARTIFICIAL INTELLIGENCE', 'CLOUD', 'INNV', 'TECH'
    ]):
        return 'Alternatives'

    # Fixed income - include abbreviations
    if any(x in name_upper for x in [
        'GILT', 'BOND', 'BD ', ' BD', 'TREASURY', 'TREASURIES', 'TSRY', 'TRES ',
        'TRSRY', 'SOVEREIGN', 'SOV ', 'CREDIT', 'INFLATION', 'TIPS', 'CORPORATE',
        'CORP ', ' CORP', 'HIGH YIELD', ' HY ', 'HYG', 'FLOATING RATE',
        'GOVERNMENT', 'GOV ', 'AGGREGATE', ' AGG', 'IBOXX', 'ITRAXX',
        'MORTGAGE', 'MTGE', 'FIXED INCOME', 'ULTRASHORT', 'ULTRASRT',
        'TBILL', 'T-BILL', 'TREAS', ' IG ', 'INV GR', 'INFL LNK',
        'GVT ', ' GVT', 'GOVT '
    ]):
        return 'Fixed Income'

    # Cash
    if any(x in name_upper for x in ['OVERNIGHT', 'MONEY MARKET', 'CASH', 'LIQUIDITY']):
        return 'Cash'

    # UK Equity
    if any(x in name_upper for x in ['FTSE 100', 'FTSE100', 'UK EQ', 'UNITED KINGDOM']):
        return 'UK Equity'

    # Global Equity - include abbreviations
    if any(x in name_upper for x in [
        'EQUITY', 'EQUITIES', 'EQTY', ' EQ ', 'MSCI', 'S&P', 'SP500', 'FTSE',
        'NASDAQ', 'STOXX', 'STOCK', 'SHARES', 'WORLD', 'GLOBAL', 'EMERGING',
        'RUSSELL', 'NIKKEI', 'TOPIX', 'DAX', 'CAC', 'JAPAN', 'CHINA', 'INDIA',
        'EUROPE', 'ASIA', 'DIVIDEND', 'INCOME', 'GROWTH', 'VALUE', 'MOMENTUM',
        'QUALITY', 'FACTOR', 'MIN VAR', 'SMALL CAP', 'SM CAP', 'MID CAP',
        'LARGE CAP', 'LG CAP', 'DOW JONES', 'NIFTY', 'BIOTECH', 'BANKS',
        'HEALTHCARE', 'UTILITIES', 'INDUSTRIALS', 'CONSUMER', 'COMMUNICATIONS',
        'MATERIALS', 'MORNINGSTAR', 'ACWI', 'EME ', 'EMRG', 'MULTI',
        'SUSTAIN', 'ESG', 'SRI', 'ACTIVEBETA', 'ALPHADEX', 'HSCEI', 'CSI',
        'STOXX', 'EUROSTOXX', 'EURO STOXX', 'RUSSELL', 'VOLATILITY', 'VOL'
    ]):
        return 'Global Equity'

    return 'N/A'

etfs['Asset_Class'] = etfs['Long Name'].apply(map_asset_class_v3)
print(etfs['Asset_Class'].value_counts())
print(f"\nRemaining N/A: {(etfs['Asset_Class'] == 'N/A').sum()}")


na_etfs = etfs[etfs['Asset_Class'] == 'N/A']
print(na_etfs[['Mnemonic', 'Long Name']].to_string())

def map_asset_class_final(name):
    name_upper = str(name).upper()

    # Currency ETFs
    if any(x in name_upper for x in ['LONG GBP', 'SHORT GBP', 'LONG EUR', 'SHORT EUR', 
        'LONG USD', 'SHORT USD', 'LONG JPY', 'SHORT JPY', 'LONG CNY', 'SHORT CNY',
        'LONG CHF', 'SHORT CHF', 'RATE SWAP', 'CURVE STEEP', 'CURVE FLAT']):
        return 'Alternatives'

    # Leveraged/short
    if any(x in name_upper for x in ['LEVERAGE', '2X', '3X', '5X', 'DAILY SHORT', 'DAILY LONG']):
        return 'Alternatives'

    # Commodities
    if any(x in name_upper for x in [
        'GOLD', 'SILVER', 'COPPER', 'NICKEL', 'ZINC', 'ALUMINIUM', 'PLATINUM',
        'PALLADIUM', 'RHODIUM', 'TIN', 'LEAD', 'COMMODITY', 'COMMODITIES', 'COMMO',
        'AGRICULTURE', 'GRAINS', 'WHEAT', 'CORN', 'SUGAR', 'COFFEE', 'COCOA',
        'COTTON', 'SOYBEANS', 'SOFTS', 'NATURAL GAS', 'GASOLINE', 'PETROLEUM',
        'OIL', 'BRENT', 'WTI', 'METAL', 'MINING', 'LIVESTOCK', 'CATTLE', 'HOGS',
        'REIT', 'PROPERTY', 'PPTY', 'PROP ', 'REAL EST', 'INFRASTRUCTURE', 'TIMBER',
        'BLOCKCHAIN', 'CANNABIS', 'MLP', 'ROBOTICS', 'MANAGED FUTURES',
        'CMCI', 'CRB', 'RICI', 'BCOM', 'GSCI', 'ENERGY',
        'CYBER', 'BATTERY', 'ECOMMERCE', 'PHARMA', 'ROLL SELECT', 'CMDT'
    ]):
        return 'Alternatives'

    # Fixed income - final sweep of abbreviations
    if any(x in name_upper for x in [
        'GILT', ' BOND', 'BD ', ' BD', 'TREASURY', 'TREASURIES', 'TSRY', 'TRES ',
        'TRSRY', 'TSY ', ' TSY', 'SOVEREIGN', 'SOV ', 'CREDIT', 'INFLATION', 'INFLA',
        'TIPS', 'CORPORATE', 'CORP ', ' CORP', 'CRP ', ' CRP', 'HIGH YIELD', ' HY ',
        'FLOATING RATE', 'FLOAT RATE', 'GOVERNMENT', 'GOV ', ' GOV', 'AGGREGATE',
        ' AGG', 'IBOXX', 'ITRAXX', 'MORTGAGE', 'MTGE', 'FIXED INCOME', 'ULTRASHORT',
        'TBILL', 'T-BILL', 'TREAS', ' IG ', 'INV GR', 'INFL LNK', 'GVT ', ' GVT',
        'GOVT ', 'SOVER', 'EMMA SOV', 'LIQ CRP', 'SHORT MAT', 'LQD', 'HGD',
        ' BND', 'BND ', 'TR BND', 'T BND', 'EM BND', 'CRP BND'
    ]):
        return 'Fixed Income'

    # Cash
    if any(x in name_upper for x in ['OVERNIGHT', 'MONEY MARKET', 'CASH', 'LIQUIDITY',
                                       'SHORT MATURITY', 'ULTRA SHORT']):
        return 'Cash'

    # UK Equity
    if any(x in name_upper for x in ['FTSE 100', 'FTSE100', 'UK EQ', 'UNITED KINGDOM']):
        return 'UK Equity'

    # Global Equity - final sweep
    if any(x in name_upper for x in [
        'EQUITY', 'EQUITIES', 'EQTY', ' EQ ', 'MSCI', 'S&P', 'SP500', 'FTSE',
        'NASDAQ', 'STOXX', 'STOCK', 'SHARES', 'WORLD', 'GLOBAL', 'EMERGING',
        'RUSSELL', 'NIKKEI', 'TOPIX', 'DAX', 'CAC', 'JAPAN', 'CHINA', 'INDIA',
        'EUROPE', 'ASIA', 'DIVIDEND', 'INCOME', 'GROWTH', 'VALUE', 'MOMENTUM',
        'QUALITY', 'FACTOR', 'MIN VAR', 'SMALL CAP', 'SM CAP', 'MID CAP',
        'LARGE CAP', 'LG CAP', 'DOW JONES', 'NIFTY', 'BIOTECH', 'BANKS',
        'HEALTHCARE', 'UTILITIES', 'INDUSTRIALS', 'CONSUMER', 'COMMUNICATIONS',
        'MATERIALS', 'MORNINGSTAR', 'ACWI', 'EME ', 'EMRG', 'SUSTAIN', 'ESG',
        'SRI', 'ACTIVEBETA', 'ALPHADEX', 'HSCEI', 'CSI', 'EUROSTOXX',
        'VOLATILITY', 'CAPE', 'ISEQ', 'ISRAEL', 'AUSTRALIA', 'NIK',
        'DIGITALISATION', 'DGTL', 'WATER', 'CLEAN', 'AGEING', 'DIVERSITY',
        'INNOVATION', 'INNOV', 'TECH', 'AI ', 'INFR', 'VIDEO', 'INTERNET',
        'MULTI', 'FAC MIX', 'SMCAP', 'PRIME EUR', 'PRIME USA', 'PRIME JAP',
        'STRENGTH', 'LIBERTA', 'LIBERTY', 'SUSTAIN', 'RESEARC', 'ENHANCED'
    ]):
        return 'Global Equity'

    return 'N/A'

etfs['Asset_Class'] = etfs['Long Name'].apply(map_asset_class_final)
print(etfs['Asset_Class'].value_counts())
print(f"\nRemaining N/A: {(etfs['Asset_Class'] == 'N/A').sum()}")

na_etfs = etfs[etfs['Asset_Class'] == 'N/A']
print(na_etfs[['Mnemonic', 'Long Name']].to_string())

# Manual override for final 29 uncategorised ETFs
manual_overrides = {
    'U37G': 'Fixed Income',    # LYX CORE $ T 3-7Y - US Treasuries
    'US37': 'Fixed Income',    # same
    'DESD': 'Global Equity',   # WISDOMTREE US SMALLCAP DIV
    'DESE': 'Global Equity',   # same
    'EGRG': 'Global Equity',   # WISDOMTREE EZ QD GROW
    'EGRP': 'Global Equity',   # same
    'EGRA': 'Global Equity',   # same
    'EGRW': 'Global Equity',   # same
    'DEMS': 'Global Equity',   # WT EM EI - Emerging Markets Equity Income
    'DEMR': 'Global Equity',   # same
    'BGX':  'Global Equity',   # EXPAT BULGARIA SOFIX - Eastern European equity
    'MDBG': 'Fixed Income',    # UBS ETF SDBB - Swiss bond
    'SGQX': 'Global Equity',   # LYXOR SGQI - Quality Income equity
    'DHSG': 'Global Equity',   # WISDOMTREE USE ETF GBP Hedged - US equity hedged
    'UBXX': 'Fixed Income',    # UBS ETF JPM USD EM DIV 1-5 - EM bonds
    'EMLO': 'Fixed Income',    # UBS ETF JPM EM - EM bonds
    'AIAG': 'Global Equity',   # LG ARTIFICIAL INTELLIGENCE ETF
    'AIAI': 'Global Equity',   # same
    'LUMV': 'Global Equity',   # OSSIAM US MINIMUM VARIANCE - equity
    'USMV': 'Global Equity',   # same
    'IGSG': 'Global Equity',   # ISHR DW JONES GLB SUST SCRNED - equity
    'IESG': 'Global Equity',   # same
    'IGSU': 'Global Equity',   # same
    'IMBA': 'Fixed Income',    # IS US MORTGAGE BACKED SEC
    'RDXS': 'Global Equity',   # INVESCO RDX - Russian equity index
    'XGLF': 'Global Equity',   # X GCC SELECT - Gulf equity
    'RUSB': 'Fixed Income',    # ITI RUSFOC USD EB - Russian bonds
    'UESD': 'Fixed Income',    # ISH GBP ULTRA BOND ESG
    'TVOU': 'Alternatives',    # TAB JPM GLOBAL VOLATILITY - vol strategy
}

# Apply overrides
for mnemonic, asset_class in manual_overrides.items():
    etfs.loc[etfs['Mnemonic'] == mnemonic, 'Asset_Class'] = asset_class

print(etfs['Asset_Class'].value_counts())
print(f"\nRemaining N/A: {(etfs['Asset_Class'] == 'N/A').sum()}")

etfs.to_csv(r'D:\Data\Asset Allocation Project\lse_etfs_categorised.csv', index=False)
print("Saved!")
print(etfs[['Ticker', 'Long Name', 'Asset_Class']].head(10))


# Reload FTSE 100
ftse100 = pd.read_csv(r'D:\Data\Asset Allocation Project\ftse100.csv')

# Rename columns to match ETF file
ftse100 = ftse100.rename(columns={'Symbol': 'Ticker', 'Name': 'Long Name'})

# Add missing columns to match ETF file
ftse100['Issuer Name'] = 'N/A'
ftse100['Security Type'] = 'UK Stock'
ftse100['Mnemonic'] = ftse100['Ticker'].str.replace('.L', '', regex=False)

# Reorder columns to match etfs DataFrame
ftse100 = ftse100[['Mnemonic', 'Long Name', 'Issuer Name', 'Security Type', 'Ticker', 'Asset_Class']]

print(f"FTSE 100 shape: {ftse100.shape}")
print(ftse100.head())

# Merge ETFs and FTSE 100 into one master DataFrame
master = pd.concat([etfs, ftse100], ignore_index=True)

# Sense check
print(f"ETFs: {len(etfs)}")
print(f"FTSE 100: {len(ftse100)}")
print(f"Master total: {len(master)}")
print(f"\nAsset Class breakdown:")
print(master['Asset_Class'].value_counts())

# Save master file
master.to_csv(r'D:\Data\Asset Allocation Project\master_securities.csv', index=False)
print("\nMaster file saved!")


# Assign Geography based on Long Name keywords
def map_geography(name, asset_class):
    name_upper = str(name).upper()
    
    # Cash and short duration - no geography needed
    if asset_class == 'Cash':
        return 'Cash'
    
    # UK
    if any(x in name_upper for x in [
        'UK', 'U.K', 'UNITED KINGDOM', 'FTSE 100', 'FTSE100', 'FTSE 250',
        'FTSE250', 'GILT', 'GILTS', 'BRITISH', 'BRITAIN', 'STERLING',
        'GBP', 'FTSE ALL SHARE', 'FTSE ACTUARIES'
    ]):
        return 'UK'
    
    # US
    if any(x in name_upper for x in [
        'US ', ' US ', 'U.S', 'USA', 'S&P', 'SP500', 'NASDAQ', 'DOW JONES',
        'RUSSELL', 'TREASURY', 'TREASURIES', 'TIPS', 'DOLLAR', 'USD',
        'AMERICAN', 'AMERICA', 'US CORP', 'US EQUITY', 'US STOCK'
    ]):
        return 'US'
    
    # Europe
    if any(x in name_upper for x in [
        'EUROPE', 'EUROPEAN', 'EURO', 'EUR', 'STOXX', 'DAX', 'CAC',
        'EUROZONE', 'EMU', 'GERMANY', 'FRANCE', 'ITALY', 'SPAIN',
        'NORDIC', 'SCANDINAV'
    ]):
        return 'Europe'
    
    # Japan
    if any(x in name_upper for x in [
        'JAPAN', 'JAPANESE', 'NIKKEI', 'TOPIX', 'JPX', 'JPY'
    ]):
        return 'Japan'
    
    # China
    if any(x in name_upper for x in [
        'CHINA', 'CHINESE', 'CSI', 'HSCEI', 'A-SHARE', 'RENMINBI', 'CNY'
    ]):
        return 'China'
    
    # Asia Pacific
    if any(x in name_upper for x in [
        'ASIA', 'ASIAN', 'PACIFIC', 'AUSTRALIA', 'KOREA', 'TAIWAN',
        'SINGAPORE', 'HONG KONG', 'INDIA', 'INDIAN'
    ]):
        return 'Asia Pacific'
    
    # Emerging Markets
    if any(x in name_upper for x in [
        'EMERGING', 'EMERG', 'EM ', ' EM', 'FRONTIER', 'DEVELOPING',
        'BRIC', 'LATAM', 'LATIN AMERICA', 'BRAZIL', 'RUSSIA', 'AFRICA'
    ]):
        return 'Emerging Markets'
    
    # Global
    if any(x in name_upper for x in [
        'GLOBAL', 'WORLD', 'INTERNATIONAL', 'ACWI', 'ALL WORLD',
        'ALL-WORLD', 'WORLDWIDE', 'MULTI', 'BROAD'
    ]):
        return 'Global'
    
    # UK stocks default to UK
    if asset_class == 'UK Equity':
        return 'UK'
    
    # Default
    return 'Global'

master['Geography'] = master.apply(
    lambda row: map_geography(row['Long Name'], row['Asset_Class']), axis=1
)

print(master['Geography'].value_counts())
print(f"\nTotal: {len(master)}")

# Save updated master
master.to_csv(r'D:\Data\Asset Allocation Project\master_securities.csv', index=False)
print("Master file updated with Geography!")

Issuer Name
ISHARES IV PLC                         129
MULTI UNITS LUXEMBOURG                 127
LEVERAGE SHARES PUBLIC LIMITED CO.     120
XTRACKERS                              104
ISHARES II PLC                          86
                                      ... 
EXPAT ASSET MANAGEMENT EAD               1
DB ETC INDEX PLC                         1
WISDOMTREE HEDGED METAL SECURITIES       1
AMUNDI PHYSICAL METALS PLC               1
FINEX FUNDS ICAV                         1
Name: count, Length: 68, dtype: int64
Asset_Class
Global Equity    1000
N/A               618
Alternatives      434
Fixed Income      228
UK Equity          20
Cash                5
Name: count, dtype: int64

N/A count: 618
Total N/A: 618
0    Mnemonic                                 Long Name Asset_Class
7        SPAL            INVESCO PHYSICAL PALLADIUM ETC         N/A
8        SPPT             INVESCO PHYSICAL PLATINUM ETC         N/A
57       SPAP            INVESCO PHYSICAL PALLADIUM ETC         N/A
58  